In [52]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, TimeSeriesSplit
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path

import sys
sys.path.append("../src")
from time_utils import TimeCleaner
from gap_splitter import LargeGapSplitter

In [53]:
# Display options
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)
DATA_PATH = Path("../Data/Original_Data/EnergyParameterDataset.xlsx")
data = pd.read_excel(DATA_PATH)

In [54]:
MOTOR_MAP = {
    "YWNC2_CONE": ["YWNC-203", "YWNC2 CONE"],
    "YWNC2_CUP":  ["YWNC-205", "YWNC2 CUP"],
    "YWNC3_CONE": ["YWNC-303", "YWNC3 CONE"],
    "YWNC3_CUP":  ["YWNC-305", "YWNC3 CUP"],
}

dfs = {
    name: data[data["Type"].isin(types)].copy()
    for name, types in MOTOR_MAP.items()
}

# Sanity check
for k, df in dfs.items():
    print(k, df.shape)

YWNC2_CONE (47879, 7)
YWNC2_CUP (48183, 7)
YWNC3_CONE (47723, 7)
YWNC3_CUP (46463, 7)


In [55]:
dfs['YWNC2_CONE'].head()

,Time,TOTAL_NET_KWH,AVG_CURRENT,AVG_V_LL,AVG_V_LN,FREQUENCY,Type
0,"22/07/2025, 20:55:44",15422.430,2.445,411.948,237.957,50.024,YWNC-203
1,"22/07/2025, 21:00:48",15422.520,2.445,411.544,237.618,50.005,YWNC-203
2,"22/07/2025, 21:06:00",15422.613,2.428,411.182,237.445,50.008,YWNC-203
3,"22/07/2025, 21:11:06",15422.703,2.445,412.419,238.222,49.982,YWNC-203
4,"22/07/2025, 21:20:52",15422.779,0.445,411.297,237.482,50.031,YWNC-203


In [56]:
''' 
1.) Drop the duplicates values

2.) In our data, there are (' 24:) in place of (' 00:), replace it convert the data type into datetime type data
, formate in "%d/%m/%Y, %H:%M:%S" formate, sort it and set it in index.

3.) Creating the new column "KWH_diff" to store the energy consumption during the given time stamp.

2.) If there are time gap between current and previous rows more than 1 hr, make a new row between them and
assign the KWH_diff/2 value to the current row and the new formatted row. By doing this we are distributing the
KWH value which was aggregated to one point. 

'''


def preprocess_raw_df(df):
    df = df.drop_duplicates()

    df = TimeCleaner(df).clean()
    df["Time"] = pd.to_datetime(df["Time"])
    df = df.set_index("Time")
    df.index.name = "Time"

    df["KWH_diff"] = df["TOTAL_NET_KWH"].diff()
    df["KWH_diff"] = df["KWH_diff"].where(df["KWH_diff"] >= 0)
    df = df.dropna(subset=["KWH_diff"])
    
    df = LargeGapSplitter(df, threshold_hours=1.0).run()

    return df


dfs = {k: preprocess_raw_df(df) for k, df in dfs.items()}


In [58]:
hourly_dfs = {
    k: (
        df.resample("1H")
          .agg(
              HOURLY_KWH=("KWH_diff", "sum"),
              AVG_CURRENT=("AVG_CURRENT", "mean"),
              AVG_V_LN=("AVG_V_LN", "mean"),
          )
          .sort_index()
    )
    for k, df in dfs.items()
}


C:\Users\devan\AppData\Local\Temp\ipykernel_10088\781352827.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.resample("1H")
C:\Users\devan\AppData\Local\Temp\ipykernel_10088\781352827.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.resample("1H")
C:\Users\devan\AppData\Local\Temp\ipykernel_10088\781352827.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.resample("1H")
C:\Users\devan\AppData\Local\Temp\ipykernel_10088\781352827.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.resample("1H")


In [59]:
hourly_dfs['YWNC2_CONE'].head()

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN
2025-07-22 21:00:00,0.507,1.151818,234.739273
2025-07-22 22:00:00,1.197,2.508750,233.409417
2025-07-22 23:00:00,1.242,2.655833,234.676667
2025-07-23 00:00:00,1.901,3.879091,234.648636
2025-07-23 01:00:00,1.187,2.545000,235.833000


In [60]:
# # Rename the energy consumption column
for k, df in hourly_dfs.items():
    df = df.rename(columns={'KWH_diff': 'HOURLY_KWH'})
    # df['HOURLY_KWH'] = df['HOURLY_KWH'].replace(0.0, np.nan)
    hourly_dfs[k] = df

In [61]:
hourly_dfs['YWNC2_CONE'].head()

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN
2025-07-22 21:00:00,0.507,1.151818,234.739273
2025-07-22 22:00:00,1.197,2.508750,233.409417
2025-07-22 23:00:00,1.242,2.655833,234.676667
2025-07-23 00:00:00,1.901,3.879091,234.648636
2025-07-23 01:00:00,1.187,2.545000,235.833000


In [62]:
hourly_dfs['YWNC2_CONE'][hourly_dfs['YWNC2_CONE']['HOURLY_KWH'] == 0.0]

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN


In [65]:
for k, df in hourly_dfs.items():
    print(f"{k}:- \n {df.isnull().sum()}\n")

YWNC2_CONE:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64

YWNC2_CUP:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64

YWNC3_CONE:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64

YWNC3_CUP:- 
 HOURLY_KWH     0
AVG_CURRENT    0
AVG_V_LN       0
dtype: int64



In [64]:
for k, df in hourly_dfs.items():
    cols = ['AVG_CURRENT', 'AVG_V_LN']
    for col in cols:
        df[col] = df[col].fillna(df[col].median())

In [66]:
def add_features(df):
    df = df.copy()

    df["power_proxy"] = df["AVG_CURRENT"] * df["AVG_V_LN"]
    
    df["hour"] = df.index.hour
    df["weekday"] = df.index.weekday

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

    for lag in [24, 168]:
        df[f"kwh_lag_{lag}"] = df["HOURLY_KWH"].shift(lag)

    df["kwh_roll_2"] = df["HOURLY_KWH"].rolling(2).mean()
    df["kwh_roll_24"] = df["HOURLY_KWH"].rolling(24).mean()

    df["long_gap_flag"] = df["AVG_CURRENT"].isna() & df["AVG_V_LN"].isna()
    gap_id = df["long_gap_flag"].cumsum()
    df["time_since_gap"] = df.groupby(gap_id).cumcount()
    df["time_since_gap_log"] = np.log1p(df["time_since_gap"])

    baseline = df["HOURLY_KWH"].rolling(24, min_periods=12).mean()
    df["low_activity_detected"] = (df["HOURLY_KWH"] < 0.25 * baseline).astype(int)

    rolling_median = df["HOURLY_KWH"].rolling(24, min_periods=12).median()
    mad = (
        (df["HOURLY_KWH"] - rolling_median)
        .abs()
        .rolling(24, min_periods=12)
        .median()
    )
    df["spike_detected"] = (
        df["HOURLY_KWH"] > (rolling_median + 3 * mad)
    ).astype(int)

    return df

In [68]:
hourly_dfs = {k: add_features(df) for k, df in hourly_dfs.items()}

In [72]:
print(hourly_dfs.keys())

dict_keys(['YWNC2_CONE', 'YWNC2_CUP', 'YWNC3_CONE', 'YWNC3_CUP'])


In [73]:
# 1. Add TYPE column and collect DataFrames
dfs_with_type = []

for type_name, df in hourly_dfs.items():
    temp_df = df.copy()
    temp_df['TYPE'] = type_name
    dfs_with_type.append(temp_df)

# 2. Concatenate all DataFrames
final_df = pd.concat(dfs_with_type, axis=0)


In [76]:
final_df = final_df.dropna()

In [78]:
# 3. Export to CSV
final_df.to_csv(r"../Data/HourlyTransformedData/hourly_energy_all_types.csv")
final_df.to_pickle(r"../Data/HourlyTransformedData/hourly_energy_all_types.pkl")